In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2009_2010"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "DIQ_F.xpt"
csv_file = DATA_DIR / "DIQ_F_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve original pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 75)
print("NHANES 2009-2010 DIQ_F XPT -> CSV")
print("=" * 75)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "DIQ010",
    "DID040",
    "DIQ160",
    "DIQ170",
    "DIQ180",
    "DIQ050",
    "DID060",
    "DIQ060U",
    "DIQ070",
    "DIQ230",
    "DIQ240",
    "DID250",
    "DID260",
    "DIQ260U",
    "DID341",
    "DID350",
    "DIQ350U",
    "DIQ360",
    "DIQ080"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "DIQ_F column names/order differ from expected.\n"
        f"Actual: {df.columns.tolist()}"
    )


if len(df) != 10109:

    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 20:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 10,109 rows x 20 columns."
)


# ============================================================
# 4. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# Never globally replace it without validating the
# exact variable/count pattern.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


print("\n--- ORIGINAL XPT TINY-ZERO CHECK ---")


print(
    "Tiny artifact occurrences:",
    f"{tiny_total:,}"
)


print(
    "\nTiny values by variable:"
)


print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 5. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Validated against:
#
#   attached DIQ_F.xpt
#   CDC 2009-2010 DIQ_F dictionary
#
# CDC processing notes explicitly state:
#
# DID250 = 0 -> None
# DID260 = 0 -> Never
# DID341 = 0 -> None
# DID350 = 0 -> None
#

EXPECTED_TINY_COUNTS = {
    "DID250": 10,
    "DID260": 131,
    "DID341": 202,
    "DID350": 151
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated DIQ_F pattern.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if tiny_total != 494:

    raise ValueError(
        f"Unexpected total tiny-value count: {tiny_total}"
    )


print(
    "\nPASS: Exact 494-value tiny-zero pattern validated."
)


# ============================================================
# 6. RESTORE ONLY VALIDATED ZERO VALUES
# ============================================================

print("\n--- ZERO RESTORATION ---")


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    tiny_mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        tiny_mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: unexpected tiny-value count."
        )


    df.loc[
        tiny_mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 494:

    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zeros restored."
)


# ============================================================
# 7. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# This represents exactly what the corrected dataset
# should contain.
#
# ONLY the four validated variables are changed:
#
# tiny artifact -> numeric zero
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col].eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 8. CHECK FOR GENUINE DECIMAL VALUES
# ============================================================

#
# CRITICAL:
#
# Never round genuine decimal variables.
#
# This check MUST happen before Int64 conversion.
#
# DIQ_F contains no genuine fractional values.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional_mask.any():

        fractional_columns.append(
            col
        )

        print(
            col,
            "fractional values:",
            int(
                fractional_mask.sum()
            )
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal values detected. "
        "They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional values "
    "exist in DIQ_F."
)


# ============================================================
# 9. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
    or
    df["SEQN"].min()
    != 51624
    or
    df["SEQN"].max()
    != 62160
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 10. FORMAT QUESTIONNAIRE FIELDS WITHOUT .0
# ============================================================

#
# After tiny-zero correction and decimal validation,
# all DIQ_F variables are proven whole-number fields.
#
# NO .round() is used.
#

for col in df.columns:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_number_check = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not np.all(
            whole_number_check
        ):

            raise ValueError(
                f"{col} contains genuine decimals "
                "and cannot safely be converted to Int64."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All DIQ_F fields formatted "
    "without unnecessary .0."
)


# ============================================================
# 11. CDC CHECK - DIQ010
# ============================================================

#
# Doctor told you have diabetes
#
# 1 = Yes           739
# 2 = No          9,230
# 3 = Borderline    134
# 7 = Refused         1
# 9 = Don't know      5
#

EXPECTED_DIQ010 = {
    1: 739,
    2: 9230,
    3: 134,
    7: 1,
    9: 5
}


actual = (
    df["DIQ010"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_DIQ010
    or
    int(
        df["DIQ010"]
        .isna()
        .sum()
    )
    != 0
):

    raise ValueError(
        "DIQ010 differs from CDC."
    )


print(
    "PASS: DIQ010 matches CDC."
)


# ============================================================
# 12. CDC CHECK - DID040
# ============================================================

#
# Age first told had diabetes
#
# 2-79 = 725
# 80   = 11
# 999  = 3
# Missing = 9,370
#

s = df["DID040"]


if not (
    int(
        s.between(
            2,
            79
        ).sum()
    )
    == 725
    and
    int(
        s.eq(80)
        .sum()
    )
    == 11
    and
    int(
        s.eq(666)
        .sum()
    )
    == 0
    and
    int(
        s.eq(777)
        .sum()
    )
    == 0
    and
    int(
        s.eq(999)
        .sum()
    )
    == 3
    and
    int(
        s.isna()
        .sum()
    )
    == 9370
):

    raise ValueError(
        "DID040 differs from CDC."
    )


print(
    "PASS: DID040 matches CDC."
)


# ============================================================
# 13. CDC CATEGORICAL CHECKS
# ============================================================

CDC_CATEGORICAL = {

    "DIQ160": {
        "values": {
            1: 262,
            2: 6420,
            9: 10
        },
        "missing": 3417
    },

    "DIQ170": {
        "values": {
            1: 738,
            2: 6056,
            9: 28
        },
        "missing": 3287
    },

    "DIQ180": {
        "values": {
            1: 2794,
            2: 3805,
            7: 1,
            9: 222
        },
        "missing": 3287
    },

    "DIQ050": {
        "values": {
            1: 218,
            2: 9890,
            7: 1
        },
        "missing": 0
    },

    "DIQ060U": {
        "values": {
            1: 30,
            2: 183
        },
        "missing": 9896
    },

    "DIQ070": {
        "values": {
            1: 588,
            2: 545,
            9: 2
        },
        "missing": 8974
    },

    "DIQ230": {
        "values": {
            1: 319,
            2: 53,
            3: 71,
            4: 68,
            5: 227,
            9: 1
        },
        "missing": 9370
    },

    "DIQ240": {
        "values": {
            1: 584,
            2: 155
        },
        "missing": 9370
    },

    "DIQ260U": {
        "values": {
            1: 398,
            2: 130,
            3: 44,
            4: 29
        },
        "missing": 9508
    },

    "DIQ350U": {
        "values": {
            1: 408,
            2: 125,
            3: 34,
            4: 11
        },
        "missing": 9531
    },

    "DIQ360": {
        "values": {
            1: 94,
            2: 384,
            3: 86,
            4: 99,
            5: 66,
            9: 5
        },
        "missing": 9375
    },

    "DIQ080": {
        "values": {
            1: 133,
            2: 594,
            9: 7
        },
        "missing": 9375
    }
}


print("\n--- CDC CATEGORICAL CHECKS ---")


for col, expected in CDC_CATEGORICAL.items():

    actual_values = (
        df[col]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[col]
        .isna()
        .sum()
    )


    if actual_values != expected["values"]:

        raise ValueError(
            f"{col} frequencies differ from CDC.\n"
            f"Expected: {expected['values']}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected["missing"]:

        raise ValueError(
            f"{col} missing count differs from CDC.\n"
            f"Expected: {expected['missing']}\n"
            f"Actual:   {actual_missing}"
        )


    print(
        f"PASS: {col} matches CDC."
    )


# ============================================================
# 14. CDC CHECK - DID060
# ============================================================

#
# How long taking insulin
#
# 1-40 = 213
# 666  = less than 1 month = 4
# 999  = don't know = 1
# Missing = 9,891
#

s = df["DID060"]


if not (
    int(
        s.between(
            1,
            40
        ).sum()
    )
    == 213
    and
    int(
        s.eq(666)
        .sum()
    )
    == 4
    and
    int(
        s.eq(777)
        .sum()
    )
    == 0
    and
    int(
        s.eq(999)
        .sum()
    )
    == 1
    and
    int(
        s.isna()
        .sum()
    )
    == 9891
):

    raise ValueError(
        "DID060 differs from CDC."
    )


print(
    "PASS: DID060 matches CDC."
)


# ============================================================
# 15. CDC CHECK - DID250
# ============================================================

#
# Times seen doctor during past year
#
# 1-150 = 574
# 0     = None = 10
# Missing = 9,525
#
# The 10 raw tiny values represent zero.
#

s = df["DID250"]


print("\n--- DID250 ---")


print(
    "Zero:",
    int(
        s.eq(0)
        .sum()
    )
)


if not (
    int(
        s.between(
            1,
            150
        ).sum()
    )
    == 574
    and
    int(
        s.eq(0)
        .sum()
    )
    == 10
    and
    int(
        s.eq(7777)
        .sum()
    )
    == 0
    and
    int(
        s.eq(9999)
        .sum()
    )
    == 0
    and
    int(
        s.isna()
        .sum()
    )
    == 9525
):

    raise ValueError(
        "DID250 differs from CDC / "
        "zero restoration."
    )


print(
    "PASS: DID250 matches CDC."
)


# ============================================================
# 16. CDC CHECK - DID260
# ============================================================

#
# How often check blood glucose
#
# 1-15 = 601
# 0    = Never = 131
# 666  = Unable due to blindness = 7
# Missing = 9,370
#

s = df["DID260"]


print("\n--- DID260 ---")


print(
    "Zero:",
    int(
        s.eq(0)
        .sum()
    )
)


if not (
    int(
        s.between(
            1,
            15
        ).sum()
    )
    == 601
    and
    int(
        s.eq(0)
        .sum()
    )
    == 131
    and
    int(
        s.eq(666)
        .sum()
    )
    == 7
    and
    int(
        s.eq(777)
        .sum()
    )
    == 0
    and
    int(
        s.eq(999)
        .sum()
    )
    == 0
    and
    int(
        s.isna()
        .sum()
    )
    == 9370
):

    raise ValueError(
        "DID260 differs from CDC / "
        "zero restoration."
    )


print(
    "PASS: DID260 matches CDC."
)


# ============================================================
# 17. CDC CHECK - DID341
# ============================================================

#
# Times doctor checked feet during past year
#
# 1-96 = 524
# 0    = None = 202
# 9999 = Don't know = 4
# Missing = 9,379
#

s = df["DID341"]


print("\n--- DID341 ---")


print(
    "Zero:",
    int(
        s.eq(0)
        .sum()
    )
)


if not (
    int(
        s.between(
            1,
            96
        ).sum()
    )
    == 524
    and
    int(
        s.eq(0)
        .sum()
    )
    == 202
    and
    int(
        s.eq(7777)
        .sum()
    )
    == 0
    and
    int(
        s.eq(9999)
        .sum()
    )
    == 4
    and
    int(
        s.isna()
        .sum()
    )
    == 9379
):

    raise ValueError(
        "DID341 differs from CDC / "
        "zero restoration."
    )


print(
    "PASS: DID341 matches CDC."
)


# ============================================================
# 18. CDC CHECK - DID350
# ============================================================

#
# How often participant checks feet
#
# 1-52 = 578
# 0    = None = 151
# 9999 = Don't know = 1
# Missing = 9,379
#

s = df["DID350"]


print("\n--- DID350 ---")


print(
    "Zero:",
    int(
        s.eq(0)
        .sum()
    )
)


if not (
    int(
        s.between(
            1,
            52
        ).sum()
    )
    == 578
    and
    int(
        s.eq(0)
        .sum()
    )
    == 151
    and
    int(
        s.eq(7777)
        .sum()
    )
    == 0
    and
    int(
        s.eq(9999)
        .sum()
    )
    == 1
    and
    int(
        s.isna()
        .sum()
    )
    == 9379
):

    raise ValueError(
        "DID350 differs from CDC / "
        "zero restoration."
    )


print(
    "PASS: DID350 matches CDC."
)


# ============================================================
# 19. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# Expected changes from raw XPT:
#
# DID250  tiny -> 0    (10)
# DID260  tiny -> 0   (131)
# DID341  tiny -> 0   (202)
# DID350  tiny -> 0   (151)
#
# NOTHING ELSE may change.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        failures.append(
            col
        )


if failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(failures)
    )


print(
    "PASS: All 20 variables match expected "
    "corrected XPT values exactly."
)


# ============================================================
# 20. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO float_format.
# NO rounding.
#
# Missing values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 21. RE-READ EXPORTED CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 22. EXACT CORRECTED XPT -> CSV FIDELITY
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


csv_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV differences found in:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 20 CSV columns match "
    "expected corrected XPT values exactly."
)


# ============================================================
# 23. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 24. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in csv_df.select_dtypes(
    include=[np.number]
).columns:

    tiny_csv_count += int(
        csv_df[col]
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in CSV."
    )


print(
    "PASS: No tiny-value artifacts remain."
)


# ============================================================
# 25. RAW CSV .0 FORMAT CHECK
# ============================================================

#
# DIQ_F has NO genuine decimal variables.
#
# Therefore ".0" should not appear anywhere
# in this particular CSV.
#

print("\n--- RAW CSV FORMAT CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


dot_zero_count = (
    csv_text.count(".0")
)


print(
    "Occurrences of '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    raise ValueError(
        "Unexpected .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 formatting."
)


# ============================================================
# 26. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 27. FINAL CSV SEQN CHECK
# ============================================================

print("\n--- FINAL CSV SEQN CHECK ---")


if (
    csv_df["SEQN"].min()
    != 51624
    or
    csv_df["SEQN"].max()
    != 62160
    or
    int(
        csv_df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        csv_df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


print(
    "PASS: Final CSV SEQN is valid."
)


# ============================================================
# 28. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 75
)


print(
    "DIQ_F CONVERSION COMPLETE"
)


print(
    "=" * 75
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "DID250 zero restorations:",
    10
)


print(
    "DID260 zero restorations:",
    131
)


print(
    "DID341 zero restorations:",
    202
)


print(
    "DID350 zero restorations:",
    151
)


print(
    "Total validated zero restorations:",
    f"{correction_total:,}"
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "XPT -> CSV unexpected differences: 0 expected"
)


print(
    "Missing-value differences: 0 expected"
)


print(
    "Tiny values remaining: 0 expected"
)


print(
    "Unnecessary .0: 0 expected"
)


print(
    "Scientific notation: 0 expected"
)

NHANES 2009-2010 DIQ_F XPT -> CSV
Rows: 10,109
Columns: 20
PASS: Structure = 10,109 rows x 20 columns.

--- ORIGINAL XPT TINY-ZERO CHECK ---
Tiny artifact occurrences: 494

Tiny values by variable:
DID341    202
DID350    151
DID260    131
DID250     10

PASS: Exact 494-value tiny-zero pattern validated.

--- ZERO RESTORATION ---
DID250: 10 restored to 0
DID260: 131 restored to 0
DID341: 202 restored to 0
DID350: 151 restored to 0

PASS: 494 validated zeros restored.

--- FRACTIONAL VALUE CHECK ---
PASS: No genuine fractional values exist in DIQ_F.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 51624
Maximum: 62160
PASS: SEQN validated.

PASS: All DIQ_F fields formatted without unnecessary .0.
PASS: DIQ010 matches CDC.
PASS: DID040 matches CDC.

--- CDC CATEGORICAL CHECKS ---
PASS: DIQ160 matches CDC.
PASS: DIQ170 matches CDC.
PASS: DIQ180 matches CDC.
PASS: DIQ050 matches CDC.
PASS: DIQ060U matches CDC.
PASS: DIQ070 matches CDC.
PASS: DIQ230 matches CDC.
PASS: DIQ240 matches CDC.
PAS